In [1]:
import numpy as np
import cv2
import os
import os.path as osp
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn2c import SVMClassifier
# Import standard SVC for accuracy checking
from sklearn.svm import SVC 
from sklearn.metrics import accuracy_score

# --- Configuration ---
EXPORT_DIR = osp.join("exported_models", "svm_digits")
EXPORT_NAME = "svm_digits_config"
os.makedirs(EXPORT_DIR, exist_ok=True)
TRAIN_SIZE = 3000

def get_c_compatible_hu(image_28x28):
    """
    Python implementation of 'hu.c' logic.
    Ensures training data matches MCU calculation exactly.
    """
    # 1. Binarize (0.0 or 1.0)
    # MNIST is 0-255. We threshold at 128.
    img = (image_28x28 > 128).astype(float)
    rows, cols = img.shape

    # 1. Raw Moments
    y_indices, x_indices = np.mgrid[:rows, :cols]
    m00 = np.sum(img)
    if m00 == 0: return np.zeros(7)

    m10 = np.sum(x_indices * img)
    m01 = np.sum(y_indices * img)
    
    # 2. Centroid
    cx = m10 / m00
    cy = m01 / m00

    # 3. Central Moments
    def mu(p, q):
        return np.sum(((x_indices - cx)**p) * ((y_indices - cy)**q) * img)

    mu00 = m00
    # Calculate required central moments
    # (We only calculate what is needed for nu)
    mu20 = mu(2, 0)
    mu02 = mu(0, 2)
    mu11 = mu(1, 1)
    mu30 = mu(3, 0)
    mu03 = mu(0, 3)
    mu21 = mu(2, 1)
    mu12 = mu(1, 2)

    # 4. Scale Invariant (nu)
    def nu(p, q):
        return mu(p, q) / (mu00 ** ((p + q) / 2.0 + 1.0))

    n20 = nu(2, 0)
    n02 = nu(0, 2)
    n11 = nu(1, 1)
    n30 = nu(3, 0)
    n03 = nu(0, 3)
    n21 = nu(2, 1)
    n12 = nu(1, 2)

    # 5. Hu Invariants (Formulas match hu.c)
    h = np.zeros(7)
    h[0] = n20 + n02
    h[1] = (n20 - n02)**2 + 4 * n11**2
    h[2] = (n30 - 3 * n12)**2 + (3 * n21 - n03)**2
    h[3] = (n30 + n12)**2 + (n21 + n03)**2
    
    t3 = n30 + n12
    t4 = n21 + n03
    
    h[4] = (n30 - 3 * n12) * t3 * (t3**2 - 3 * t4**2) + \
           (3 * n21 - n03) * t4 * (3 * t3**2 - t4**2)
           
    h[5] = (n20 - n02) * (t3**2 - t4**2) + 4 * n11 * t3 * t4
    
    h[6] = (3 * n21 - n03) * t3 * (t3**2 - 3 * t4**2) - \
           (n30 - 3 * n12) * t4 * (3 * t3**2 - t4**2)

    # 6. LOG TRANSFORM (Essential for SVM stability)
    with np.errstate(divide='ignore', invalid='ignore'):
        h = -1 * np.sign(h) * np.log10(np.abs(h) + 1e-6)
        
    return h.astype(np.float32)

def main():
    print("Loading MNIST...")
    mnist = datasets.fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X_raw = mnist.data.astype(np.uint8)
    y_raw = mnist.target.astype(int)

    print("Extracting features (this is slow in Python, please wait)...")
    X_features = []
    
    # Process subset
    limit = TRAIN_SIZE + 500
    for i in range(limit):
        img = X_raw[i].reshape(28, 28)
        hu = get_c_compatible_hu(img)
        X_features.append(hu)
        if i % 500 == 0: print(f"  {i}/{limit}")

    X = np.array(X_features)
    y = y_raw[:limit]

    # Balance & Limit
    from sklearn.utils import shuffle
    X, y = shuffle(X, y, random_state=42)
    X, y = X[:TRAIN_SIZE], y[:TRAIN_SIZE]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # --- 1. ACCURACY CHECK (Standard Sklearn) ---
    print("\nVerifying accuracy with standard sklearn SVC...")
    check_svm = SVC(C=10.0, kernel='rbf', gamma='scale')
    check_svm.fit(X_train, y_train)
    
    y_pred = check_svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"📊 Estimated Accuracy: {acc*100:.2f}%")

    # --- 2. EXPORT (sklearn2c) ---
    print("Training exportable model (sklearn2c)...")
    # Note: We assume sklearn2c params map 1:1 to sklearn
    svm = SVMClassifier(C=10.0, kernel='rbf', gamma='scale')
    svm.train(X_train, y_train)

    # Export
    out_path = osp.join(EXPORT_DIR, EXPORT_NAME)
    print("Exporting Config...")
    svm.export(out_path)

    # --- 3. APPEND SCALER ---
    print("Appending Scaler Constants...")
    with open(out_path + ".h", "a") as f:
        f.write("\n// SCALER CONSTANTS\n")
        f.write("extern const float SCALER_MEAN[7];\n")
        f.write("extern const float SCALER_SCALE[7];\n")
        # Also append n_support if needed by C logic
        f.write("extern const int n_support[10];\n")

    with open(out_path + ".c", "a") as f:
        m_str = ", ".join([f"{x:.6f}f" for x in scaler.mean_])
        s_str = ", ".join([f"{x:.6f}f" for x in scaler.scale_])
        
        # Get n_support from the helper model (standard sklearn)
        # support_vectors_per_class = check_svm.n_support_
        n_supp_str = ", ".join(map(str, check_svm.n_support_))

        f.write(f"\n// SCALER CONSTANTS\n")
        f.write(f"const float SCALER_MEAN[7] = {{ {m_str} }};\n")
        f.write(f"const float SCALER_SCALE[7] = {{ {s_str} }};\n")
        f.write(f"const int n_support[10] = {{ {n_supp_str} }};\n")
    
    print("✅ Done. Files generated.")

if __name__ == "__main__":
    main()

Loading MNIST...
Extracting features (this is slow in Python, please wait)...
  0/3500
  500/3500
  1000/3500
  1500/3500
  2000/3500
  2500/3500
  3000/3500

Verifying accuracy with standard sklearn SVC...
📊 Estimated Accuracy: 54.83%
Training exportable model (sklearn2c)...
Exporting Config...
Appending Scaler Constants...
✅ Done. Files generated.
